# Data Validation

## Validation 1: Record Count Validation

In [0]:
bronze = spark.table("learntrack_lms_analytics.default.bronze_enrolment_activity")

silver = spark.table("learntrack_lms_analytics.default.silver_enrolment_activity")

print(f"Bronze Records : {bronze.count()}")
print(f"Silver Records : {silver.count()}")

Bronze Records : 2000
Silver Records : 1990


## Validation 2: Duplicate Check

In [0]:
duplicates = (
    silver
    .groupBy("enrolment_id")
    .count()
    .filter("count > 1")
)

display(duplicates)

enrolment_id,count


## Validation 3: Primary Key Check

In [0]:
from pyspark.sql.functions import col

silver.filter(
    col("enrolment_id").isNull()
).show()

+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days|
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+



## Validation 4: Status Validation

In [0]:
silver.select("status").distinct().show()

+-----------+
|     status|
+-----------+
|  Completed|
|In Progress|
|    Dropped|
|Not Started|
+-----------+



## Validation 5: Progress Percentage

In [0]:
silver.filter(
    (col("progress_pct") < 0) |
    (col("progress_pct") > 100)
).show()

+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days|
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+



## Validation 6: Assessment Score


In [0]:
silver.filter(
    (col("assessment_score") < 0) |
    (col("assessment_score") > 100)
).show()

+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days|
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+



## Validation 7: Certificate Issued

In [0]:
silver.groupBy("certificate_issued").count().show()

+------------------+-----+
|certificate_issued|count|
+------------------+-----+
|               Yes|  572|
|                No| 1418|
+------------------+-----+



## Validation 8: Completion Date Logic

In [0]:
silver.filter(
    (col("status") == "Completed") &
    (col("actual_completion_date").isNull())
).show()

+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
|enrolment_id|learner_id|course_id|enrol_date|expected_completion_date|actual_completion_date|status|progress_pct|last_activity_date|assessment_score|attempts|feedback_rating|certificate_issued|learning_duration_days|completion_delay_days|
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+
+------------+----------+---------+----------+------------------------+----------------------+------+------------+------------------+----------------+--------+---------------+------------------+----------------------+---------------------+



## Validation 9: Gold Table Validation

In [0]:
gold_tables = [
    "gold_course_completion",
    "gold_learner_engagement",
    "gold_instructor_performance",
    "gold_assessment_performance",
    "gold_dropout_reenrolment"
]

for table in gold_tables:
    df = spark.table(f"learntrack_lms_analytics.default.{table}")
    print(f"{table}: {df.count()} rows")

gold_course_completion: 60 rows
gold_learner_engagement: 1990 rows
gold_instructor_performance: 19 rows
gold_assessment_performance: 60 rows
gold_dropout_reenrolment: 1910 rows
